[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/InterstellarNerd8760/lunar-atlas/blob/proxima/terrain-notebooks/notebooks/prepare-textures.ipynb)

# Earth and star textures for the sky

Downloads two free NASA images and saves web-friendly JPGs for three.js:

| Texture | Source | Original | License / credit |
| --- | --- | --- | --- |
| Earth | [Blue Marble: Next Generation with Topography and Bathymetry, July 2004](https://science.nasa.gov/earth/earth-observatory/blue-marble-next-generation/base-topography-bathymetry/) | 5400×2700 JPEG (2.3 MB) or 21600×10800 JPEG (27.2 MB) | NASA imagery, not copyrighted ([NASA media guidelines](https://www.nasa.gov/nasa-brand-center/images-and-media/)). Credit: NASA Earth Observatory. |
| Stars | [Deep Star Maps 2020, NASA SVS 4851](https://svs.gsfc.nasa.gov/4851/) | 4096×2048 EXR (36.0 MB) or 8192×4096 EXR (130.5 MB), celestial (RA/Dec) plate carrée | Credit: NASA/Goddard Space Flight Center Scientific Visualization Studio. Gaia DR2: ESA/Gaia/DPAC. |

**Size advice:** from the Moon, Earth is only about 2° across, so a 2048-wide Earth is plenty for the ground view; use 4096 only if the camera flies close to Earth. The star sphere fills the whole screen, so use 8192×4096 on desktop and 4096×2048 for phones.

Free CPU runtime on Colab or Kaggle (Kaggle: turn on **Internet**). **Runtime → Run all.**

In [ ]:
%pip -q install pillow numpy requests OpenEXR

## 1. Settings

In [ ]:
EARTH_SOURCE = "5400"     # "5400" (2.3 MB download) or "21600" (27.2 MB, needed for an 8192-wide Earth)
EARTH_WIDTHS = [2048, 4096]
STAR_SOURCE = "8k"        # "4k" (36 MB download) or "8k" (130 MB)
STAR_WIDTHS = [4096, 8192]
STAR_EXPOSURE = 1.0       # >1 brightens faint stars, <1 darkens the sky
JPEG_QUALITY = 88
USE_DRIVE = False
DOWNLOAD_ZIP = True

## 2. Download

In [ ]:
import sys
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED

import numpy as np
import OpenEXR
import requests
from PIL import Image

Image.MAX_IMAGE_PIXELS = None   # the 21600-wide Earth is bigger than Pillow's default safety limit
IN_COLAB = "google.colab" in sys.modules
RAW = Path("data/raw")
OUT_ROOT = Path("data/work")
if USE_DRIVE and IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_ROOT = Path("/content/drive/MyDrive/lunar-atlas")
EXPORT = OUT_ROOT / "exports" / "textures"
RAW.mkdir(parents=True, exist_ok=True)
EXPORT.mkdir(parents=True, exist_ok=True)

BMNG = "https://assets.science.nasa.gov/content/dam/science/esd/eo/images/bmng/bmng-topography-bathymetry/july"
EARTH_URL = {"5400": f"{BMNG}/world.topo.bathy.200407.3x5400x2700.jpg",
             "21600": f"{BMNG}/world.topo.bathy.200407.3x21600x10800.jpg"}[EARTH_SOURCE]
STAR_URL = f"https://svs.gsfc.nasa.gov/vis/a000000/a004800/a004851/starmap_2020_{STAR_SOURCE}.exr"


def download(url):
    path = RAW / Path(url).name
    if not path.exists():
        part = path.with_suffix(path.suffix + ".part")
        with requests.get(url, stream=True, timeout=(15, 300)) as response:
            response.raise_for_status()
            with part.open("wb") as output:
                for chunk in response.iter_content(chunk_size=1024 * 1024):
                    output.write(chunk)
        part.replace(path)
    print(f"{path.name}: {path.stat().st_size / 1e6:.1f} MB")
    return path


earth_path = download(EARTH_URL)
star_path = download(STAR_URL)

## 3. Resize and save
The star map is stored as linear light (OpenEXR), so it gets a gamma curve to look right as a normal JPG.

In [ ]:
saved = []
earth = Image.open(earth_path).convert("RGB")
for width in EARTH_WIDTHS:
    if width > earth.width:
        print(f"Skip Earth {width}: the source is only {earth.width} wide (set EARTH_SOURCE = '21600')")
        continue
    path = EXPORT / f"earth-{width}.jpg"
    earth.resize((width, width // 2), Image.LANCZOS).save(path, quality=JPEG_QUALITY, optimize=True)
    saved.append(path)

with OpenEXR.File(str(star_path)) as exr:
    linear = np.asarray(exr.channels()["RGB"].pixels, dtype=np.float32)
srgb = np.clip(linear * STAR_EXPOSURE, 0, 1) ** (1 / 2.2)
stars = Image.fromarray((srgb * 255 + 0.5).astype(np.uint8))
del linear, srgb
for width in STAR_WIDTHS:
    if width > stars.width:
        print(f"Skip stars {width}: the source is only {stars.width} wide (set STAR_SOURCE = '8k')")
        continue
    path = EXPORT / f"stars-{width}.jpg"
    stars.resize((width, width // 2), Image.LANCZOS).save(path, quality=JPEG_QUALITY, optimize=True)
    saved.append(path)

(EXPORT / "CREDITS.txt").write_text(
    f"Earth: {EARTH_URL}\nNASA Earth Observatory, Blue Marble: Next Generation (July 2004). NASA imagery, not copyrighted.\n\n"
    f"Stars: {STAR_URL}\nNASA/Goddard Space Flight Center Scientific Visualization Studio. Gaia DR2: ESA/Gaia/DPAC.\n"
    "Star map is in celestial coordinates (RA/Dec, J2000), plate carree.\n")
for path in saved:
    print(f"{path.name}: {Image.open(path).size}, {path.stat().st_size / 1e6:.1f} MB")
display(stars.resize((1024, 512)))

## 4. Zip and download
Put the JPGs in `public/data/` (for example `public/data/earth-4096.jpg`). Keep `CREDITS.txt` with them.

In [ ]:
zip_path = OUT_ROOT / "textures.zip"
with ZipFile(zip_path, "w", compression=ZIP_DEFLATED) as bundle:
    for path in sorted(EXPORT.iterdir()):
        bundle.write(path, f"textures/{path.name}")
print(f"Zip: {zip_path.stat().st_size / 1e6:.1f} MB at {zip_path}")
if DOWNLOAD_ZIP and IN_COLAB:
    from google.colab import files
    files.download(str(zip_path))